# Momentum Crash Risk

Scales long-short momentum exposure with loser beta, market volatility and rebound indicators.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
DANIEL-MOSKOWITZ MOMENTUM CRASH RISK MODEL
==========================================

This script implements the Daniel-Moskowitz (2016) framework for identifying
momentum crash risk and dynamically managing momentum strategy exposure.

WHAT ARE MOMENTUM CRASHES?
--------------------------
Momentum strategies (long winners, short losers) work well most of the time but
suffer occasional catastrophic losses. These "crashes" occur when:
- Past losers suddenly rally strongly (mean reversion)
- Past winners decline simultaneously
- The long-short momentum portfolio collapses

Classic examples:
- 2009 Q1: +73% momentum crash (losers rallied post-crisis)
- 2001 Q2: +37% momentum crash (tech bounce after bubble)
- 1933: Depression-era loser rally

DANIEL-MOSKOWITZ INSIGHT:
-------------------------
Momentum crashes are PREDICTABLE. They occur when:

1. **Past losers have high market beta** (> 1.5)
   - Losers become increasingly risky
   - Positioned to rally when market recovers

2. **Market has declined recently** (negative returns)
   - Sets up mean reversion conditions
   - Losers "spring back" effect

3. **Market volatility is elevated**
   - Increases crash magnitude
   - Amplifies loser beta exposure

4. **Dispersion is high**
   - Winners and losers are extreme
   - Reversal potential is larger

MATHEMATICAL FRAMEWORK:
-----------------------
Standard Momentum Portfolio:
    R_MOM(t) = R_winners(t) - R_losers(t)

Crash Risk Indicator:
    CrashRisk(t) = f(β_losers(t-1), R_market(t-1), σ_market(t-1))

Dynamic Momentum Strategy:
    Exposure(t) = {
        1.0   if CrashRisk(t) < threshold_low  (full exposure)
        0.5   if threshold_low ≤ CrashRisk(t) < threshold_high  (reduced)
        0.0   if CrashRisk(t) ≥ threshold_high  (no exposure)
    }

    R_DMOM(t) = Exposure(t) * R_MOM(t)

IMPLEMENTATION STRATEGY:
------------------------
1. Construct momentum portfolios (top/bottom quintiles by past returns)
2. Calculate loser portfolio market beta dynamically
3. Track market conditions (returns, volatility)
4. Compute crash risk score
5. Dynamically scale momentum exposure
6. Backtest with and without crash risk management
7. Compare performance and drawdowns

KEY INNOVATION:
---------------
Unlike stop-losses (exit after losses), this is PREDICTIVE risk management.
We reduce exposure BEFORE crashes based on observable risk factors.
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
UNIVERSE_SIZE = 100
TICKERS_FILE = None
FORMATION_PERIOD = 252
HOLDING_PERIOD = 21
SKIP_MONTH = 21
NUM_PORTFOLIOS = 5
BETA_WINDOW = 60
MARKET_RETURN_WINDOW = 60
VOLATILITY_WINDOW = 60
BETA_THRESHOLD_HIGH = 1.5
BETA_THRESHOLD_LOW = 1.0
MARKET_RETURN_THRESHOLD = -0.05
VOLATILITY_THRESHOLD = 0.25
FULL_EXPOSURE = 1.0
REDUCED_EXPOSURE = 0.5
ZERO_EXPOSURE = 0.0
START_DATE = "2010-01-01"
END_DATE = "2024-12-31"
INITIAL_CAPITAL = 100000
COMMISSION_RATE = 0.001
BENCHMARK_TICKER = "SPY"


def fetch_stock_list(limit=100):
    """
    Fetch list of liquid stocks for universe construction.

    Parameters:
    -----------
    limit : int
        Number of stocks to fetch
        Yahoo Finance key

    Returns:
    --------
    list
        List of stock tickers

    Note:
    -----
    We focus on large-cap liquid stocks to ensure:
    - Sufficient trading history
    - Realistic momentum portfolio construction
    - Ability to actually trade these positions

    In production, would filter by:
    - Market cap > $1B
    - Average daily volume > $10M
    - Listing age > 2 years
    """
    universe = [
        "AAPL",
        "MSFT",
        "GOOGL",
        "AMZN",
        "META",
        "NVDA",
        "TSLA",
        "BRK.B",
        "JPM",
        "V",
        "JNJ",
        "WMT",
        "PG",
        "MA",
        "HD",
        "DIS",
        "PYPL",
        "NFLX",
        "ADBE",
        "CRM",
        "CMCSA",
        "PFE",
        "INTC",
        "T",
        "VZ",
        "MRK",
        "ABT",
        "NKE",
        "PEP",
        "CSCO",
        "CVX",
        "XOM",
        "KO",
        "TMO",
        "AVGO",
        "ORCL",
        "ACN",
        "MCD",
        "COST",
        "DHR",
        "TXN",
        "UNP",
        "NEE",
        "LIN",
        "MDT",
        "HON",
        "QCOM",
        "UPS",
        "AMD",
        "AMGN",
        "LOW",
        "IBM",
        "RTX",
        "BA",
        "SBUX",
        "INTU",
        "GS",
        "AXP",
        "CAT",
        "SPGI",
        "DE",
        "BLK",
        "MMM",
        "LMT",
        "BMY",
        "GILD",
        "ISRG",
        "CI",
        "MO",
        "ZTS",
        "TJX",
        "CVS",
        "USB",
        "PLD",
        "CL",
        "SO",
        "MS",
        "SYK",
        "DUK",
        "BDX",
        "CB",
        "AON",
        "APD",
        "ITW",
        "CSX",
        "MMC",
        "WM",
        "PNC",
        "NSC",
        "GD",
        "ICE",
        "EMR",
        "NOC",
        "CCI",
        "ECL",
        "TGT",
        "SHW",
        "MCO",
        "HUM",
        "GM",
        "F",
        "UBER",
    ]
    return universe[:limit]


def fetch_prices_for_universe(tickers, start_date, end_date):
    """
    Fetch price data for entire universe of stocks.

    Parameters:
    -----------
    tickers : list
        List of stock tickers
    start_date : str
        Start date
    end_date : str
        End date
        Yahoo Finance key

    Returns:
    --------
    pd.DataFrame
        DataFrame with adjusted close prices, tickers as columns

    Implementation Notes:
    ---------------------
    This is the most data-intensive step. For production:
    - Implement caching to avoid repeated API calls
    - Use batch endpoints if available
    - Handle API rate limits with delays
    - Store locally for faster subsequent runs

    NO LOOK-AHEAD BIAS: We use only data available at each point in time.
    """
    print(f"Fetching price data for {len(tickers)} stocks...")
    all_prices = {}
    failed_tickers = []
    for i, ticker in enumerate(tickers):
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" in data and len(data["historical"]) > 0:
                df = pd.DataFrame(data["historical"])
                df["date"] = pd.to_datetime(df["date"])
                df = df.sort_values("date")
                df.set_index("date", inplace=True)
                all_prices[ticker] = df["adjClose"]
                if (i + 1) % 10 == 0:
                    print(f"  Fetched {i + 1}/{len(tickers)} stocks...")
            else:
                failed_tickers.append(ticker)
        except Exception as e:
            print(f"  Error fetching {ticker}: {str(e)}")
            failed_tickers.append(ticker)
    print(f"Successfully fetched {len(all_prices)}/{len(tickers)} stocks")
    if failed_tickers:
        print(f"Failed tickers: {failed_tickers[:10]}...")
    prices_df = pd.DataFrame(all_prices)
    prices_df = prices_df.dropna(thresh=int(0.8 * len(prices_df)), axis=1)
    print(f"Final universe: {len(prices_df.columns)} stocks with sufficient data")
    return prices_df


def fetch_market_data(benchmark_ticker, start_date, end_date):
    """
    Fetch market benchmark data for beta calculation.

    Parameters:
    -----------
    benchmark_ticker : str
        Benchmark ticker (e.g., 'SPY')
    start_date : str
        Start date
    end_date : str
        End date
        Yahoo Finance key

    Returns:
    --------
    pd.Series
        Benchmark price series
    """
    print(f"\nFetching market benchmark data ({benchmark_ticker})...")
    response_prices = {"historical": download_price_records(benchmark_ticker, start_date, end_date)}
    data = response_prices
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date")
    df.set_index("date", inplace=True)
    print(f"Fetched {len(df)} days of market data")
    return df["adjClose"]


def calculate_momentum_signals(prices, formation_period, skip_month):
    """
    Calculate momentum signals for all stocks.

    Parameters:
    -----------
    prices : pd.DataFrame
        Price data for universe
    formation_period : int
        Lookback period for momentum calculation
    skip_month : int
        Days to skip to avoid short-term reversal

    Returns:
    --------
    pd.DataFrame
        Momentum scores for each stock at each date

    Methodology:
    ------------
    Standard momentum = cumulative return from t-formation_period to t-skip_month

    Example: 12-1 momentum
    - Look back 12 months (252 trading days)
    - Skip most recent month (21 days) to avoid reversal
    - Rank stocks by this return

    This is the classic Jegadeesh-Titman (1993) specification.

    NO LOOK-AHEAD BIAS: Momentum at time t uses only prices up to t-skip_month.
    """
    print(f"\nCalculating momentum signals...")
    print(f"  Formation period: {formation_period} days")
    print(f"  Skip period: {skip_month} days")
    momentum_returns = prices.shift(skip_month) / prices.shift(formation_period + skip_month) - 1
    print(f"Momentum signals calculated for {len(momentum_returns.columns)} stocks")
    return momentum_returns


def construct_momentum_portfolios(momentum_signals, num_portfolios):
    """
    Construct quintile portfolios based on momentum signals.

    Parameters:
    -----------
    momentum_signals : pd.DataFrame
        Momentum scores for each stock
    num_portfolios : int
        Number of portfolios to create (e.g., 5 for quintiles)

    Returns:
    --------
    pd.DataFrame
        Portfolio assignments (1=losers, 5=winners)

    Portfolio Construction:
    -----------------------
    At each rebalancing date:
    1. Rank all stocks by momentum score
    2. Assign to quintiles:
       - Quintile 1 (bottom 20%): Past losers
       - Quintile 5 (top 20%): Past winners
    3. Hold these portfolios until next rebalance

    The momentum strategy is:
    - Long Quintile 5 (winners)
    - Short Quintile 1 (losers)
    """
    print(f"\nConstructing {num_portfolios} momentum portfolios...")
    ranks = momentum_signals.rank(axis=1, method="first", ascending=True)
    portfolios = pd.DataFrame(
        index=momentum_signals.index, columns=momentum_signals.columns, dtype=float
    )
    for date in momentum_signals.index:
        date_momentum = momentum_signals.loc[date].dropna()
        if len(date_momentum) > 0:
            quintiles = pd.qcut(
                date_momentum,
                q=num_portfolios,
                labels=range(1, num_portfolios + 1),
                duplicates="drop",
            )
            portfolios.loc[date, quintiles.index] = quintiles.values
    print(f"Portfolios constructed")
    return portfolios


def calculate_portfolio_returns(prices, portfolio_assignments, portfolio_id):
    """
    Calculate equal-weighted returns for a specific portfolio.

    Parameters:
    -----------
    prices : pd.DataFrame
        Price data
    portfolio_assignments : pd.DataFrame
        Portfolio assignments (1-5)
    portfolio_id : int
        Which portfolio to calculate (1=losers, 5=winners)

    Returns:
    --------
    pd.Series
        Daily returns for the portfolio

    Equal-Weighted Portfolio:
    -------------------------
    At each date, invest equally in all stocks assigned to this portfolio.
    Portfolio return = average of individual stock returns.

    This is the standard academic approach. In practice, might use:
    - Value-weighted (weight by market cap)
    - Volatility-weighted (inverse volatility)
    - Rank-weighted (weight by momentum strength)
    """
    returns = prices.pct_change()
    portfolio_returns = pd.Series(index=returns.index, dtype=float)
    for date in returns.index:
        mask = portfolio_assignments.loc[date] == portfolio_id
        stocks_in_portfolio = mask[mask].index
        if len(stocks_in_portfolio) > 0:
            portfolio_returns[date] = returns.loc[date, stocks_in_portfolio].mean()
    return portfolio_returns.dropna()


def calculate_rolling_beta(portfolio_returns, market_returns, window):
    """
    Calculate rolling beta of portfolio to market.

    Parameters:
    -----------
    portfolio_returns : pd.Series
        Portfolio returns
    market_returns : pd.Series
        Market returns
    window : int
        Rolling window for beta calculation

    Returns:
    --------
    pd.Series
        Rolling beta estimates

    Beta Calculation:
    -----------------
    Beta = Cov(R_portfolio, R_market) / Var(R_market)

    Interpretation:
    - Beta = 1.0: Portfolio moves with market
    - Beta > 1.0: Portfolio amplifies market moves (risky)
    - Beta < 1.0: Portfolio dampens market moves (defensive)

    For momentum crash prediction, high loser beta (> 1.5) indicates:
    - Losers are increasingly risky
    - Positioned to rally strongly when market recovers
    - High crash risk for short position
    """
    aligned = pd.DataFrame({"portfolio": portfolio_returns, "market": market_returns}).dropna()
    rolling_cov = aligned["portfolio"].rolling(window).cov(aligned["market"])
    rolling_var = aligned["market"].rolling(window).var()
    rolling_beta = rolling_cov / rolling_var
    return rolling_beta.dropna()


def calculate_crash_risk_indicator(
    loser_beta, market_returns, market_volatility, beta_threshold, return_threshold, vol_threshold
):
    """
    Calculate momentum crash risk indicator based on Daniel-Moskowitz framework.

    Parameters:
    -----------
    loser_beta : pd.Series
        Rolling beta of loser portfolio
    market_returns : pd.Series
        Recent market returns
    market_volatility : pd.Series
        Market volatility
    beta_threshold : float
        High beta threshold
    return_threshold : float
        Negative return threshold
    vol_threshold : float
        Elevated volatility threshold

    Returns:
    --------
    pd.DataFrame
        Crash risk indicators and overall score

    Crash Risk Components:
    ----------------------
    1. High Loser Beta: β_losers > 1.5
       - Indicates losers have become very risky
       - Will rally strongly if market recovers

    2. Recent Market Decline: R_market < -5%
       - Market has declined recently
       - Sets up potential mean reversion

    3. Elevated Volatility: σ_market > 25%
       - High volatility amplifies potential crash
       - Increases magnitude of loser rally

    Overall Score:
    - 0: Low risk (all conditions favorable)
    - 1: Moderate risk (1-2 warning signals)
    - 2: High risk (2-3 warning signals)
    - 3: Extreme risk (all warning signals)
    """
    crash_risk = pd.DataFrame(index=loser_beta.index)
    crash_risk["loser_beta"] = loser_beta
    crash_risk["market_return"] = market_returns
    crash_risk["market_volatility"] = market_volatility
    crash_risk["high_beta"] = (crash_risk["loser_beta"] > beta_threshold).astype(int)
    crash_risk["negative_market"] = (crash_risk["market_return"] < return_threshold).astype(int)
    crash_risk["high_volatility"] = (crash_risk["market_volatility"] > vol_threshold).astype(int)
    crash_risk["crash_risk_score"] = (
        crash_risk["high_beta"] + crash_risk["negative_market"] + crash_risk["high_volatility"]
    )
    return crash_risk


def determine_exposure(crash_risk_score, low_threshold=1, high_threshold=2):
    """
    Determine momentum exposure based on crash risk score.

    Parameters:
    -----------
    crash_risk_score : pd.Series
        Crash risk scores (0-3)
    low_threshold : int
        Below this, full exposure
    high_threshold : int
        Above this, zero exposure

    Returns:
    --------
    pd.Series
        Exposure levels (0.0, 0.5, or 1.0)

    Exposure Rules:
    ---------------
    Score 0: Full exposure (1.0)
      - No warning signals
      - Normal momentum conditions

    Score 1: Full exposure (1.0)
      - One warning signal
      - Monitoring but not concerning

    Score 2: Reduced exposure (0.5)
      - Two warning signals
      - Elevated crash risk
      - Cut position by half

    Score 3: Zero exposure (0.0)
      - All warning signals present
      - Extreme crash risk
      - Exit momentum position entirely
    """
    exposure = pd.Series(index=crash_risk_score.index, dtype=float)
    exposure[crash_risk_score < high_threshold] = FULL_EXPOSURE
    exposure[(crash_risk_score >= high_threshold) & (crash_risk_score < 3)] = REDUCED_EXPOSURE
    exposure[crash_risk_score >= 3] = ZERO_EXPOSURE
    return exposure


def backtest_momentum_strategy(
    prices,
    portfolio_assignments,
    market_returns,
    crash_risk_data,
    use_crash_management=False,
    holding_period=21,
    commission_rate=0.001,
):
    """
    Backtest momentum strategy with optional crash risk management.

    Parameters:
    -----------
    prices : pd.DataFrame
        Price data
    portfolio_assignments : pd.DataFrame
        Portfolio assignments
    market_returns : pd.Series
        Market returns
    crash_risk_data : pd.DataFrame
        Crash risk indicators
    use_crash_management : bool
        Whether to use dynamic exposure
    holding_period : int
        Rebalancing frequency
    commission_rate : float
        Transaction costs

    Returns:
    --------
    pd.DataFrame
        Backtest results with returns, positions, and metrics

    Implementation:
    ---------------
    Long-short momentum strategy:
    - Long equal-weighted winner portfolio (quintile 5)
    - Short equal-weighted loser portfolio (quintile 1)
    - Rebalance every holding_period days
    - Apply transaction costs
    - Dynamically scale exposure based on crash risk
    """
    print(f"\nBacktesting momentum strategy...")
    print(f"  Crash risk management: {use_crash_management}")
    print(f"  Rebalancing frequency: {holding_period} days")
    returns = prices.pct_change()
    backtest = pd.DataFrame(index=returns.index)
    backtest["winners_return"] = 0.0
    backtest["losers_return"] = 0.0
    backtest["momentum_return"] = 0.0
    backtest["exposure"] = 1.0
    backtest["managed_return"] = 0.0
    current_winners = []
    current_losers = []
    days_since_rebalance = holding_period
    for i, date in enumerate(returns.index):
        if days_since_rebalance >= holding_period:
            if date in portfolio_assignments.index:
                winners_mask = portfolio_assignments.loc[date] == 5
                losers_mask = portfolio_assignments.loc[date] == 1
                current_winners = winners_mask[winners_mask].index.tolist()
                current_losers = losers_mask[losers_mask].index.tolist()
                days_since_rebalance = 0
        if len(current_winners) > 0:
            backtest.loc[date, "winners_return"] = returns.loc[date, current_winners].mean()
        if len(current_losers) > 0:
            backtest.loc[date, "losers_return"] = returns.loc[date, current_losers].mean()
        backtest.loc[date, "momentum_return"] = (
            backtest.loc[date, "winners_return"] - backtest.loc[date, "losers_return"]
        )
        if days_since_rebalance == 0:
            turnover_cost = 2 * commission_rate
            backtest.loc[date, "momentum_return"] -= turnover_cost
        if use_crash_management and date in crash_risk_data.index:
            backtest.loc[date, "exposure"] = crash_risk_data.loc[date, "exposure"]
        else:
            backtest.loc[date, "exposure"] = 1.0
        backtest.loc[date, "managed_return"] = (
            backtest.loc[date, "exposure"] * backtest.loc[date, "momentum_return"]
        )
        days_since_rebalance += 1
    backtest["momentum_cumulative"] = (1 + backtest["momentum_return"]).cumprod()
    backtest["managed_cumulative"] = (1 + backtest["managed_return"]).cumprod()
    backtest["market_cumulative"] = (
        1 + market_returns.reindex(backtest.index, fill_value=0)
    ).cumprod()
    print(f"Backtest complete")
    return backtest


def calculate_performance_metrics(returns, periods_per_year=252):
    """
    Calculate comprehensive performance metrics.

    Parameters:
    -----------
    returns : pd.Series
        Strategy returns
    periods_per_year : int
        Number of periods per year (252 for daily)

    Returns:
    --------
    dict
        Performance metrics
    """
    total_return = (1 + returns).prod() - 1
    annual_return = (1 + total_return) ** (periods_per_year / len(returns)) - 1
    annual_vol = returns.std() * np.sqrt(periods_per_year)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(periods_per_year)
    sortino = annual_return / downside_std if downside_std > 0 else 0
    win_rate = (returns > 0).sum() / len(returns)
    return {
        "Total Return": total_return,
        "Annual Return": annual_return,
        "Annual Volatility": annual_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
    }


def plot_backtest_results(backtest_results, crash_risk_data, ticker="Universe"):
    """
    Create comprehensive visualization of backtest results.
    """
    fig, axes = plt.subplots(5, 1, figsize=(15, 16))
    dates = backtest_results.index
    axes[0].plot(
        dates,
        backtest_results["momentum_cumulative"],
        label="Standard Momentum",
        color="blue",
        linewidth=2,
    )
    axes[0].plot(
        dates,
        backtest_results["managed_cumulative"],
        label="Crash-Managed Momentum",
        color="green",
        linewidth=2,
    )
    axes[0].plot(
        dates,
        backtest_results["market_cumulative"],
        label="Market (SPY)",
        color="gray",
        linewidth=1.5,
        alpha=0.7,
    )
    axes[0].set_ylabel("Cumulative Return", fontsize=11)
    axes[0].set_title(f"{ticker} Momentum Strategy Performance", fontsize=13, fontweight="bold")
    axes[0].legend(loc="upper left")
    axes[0].grid(True, alpha=0.3)
    axes[0].set_yscale("log")
    mom_cumul = backtest_results["momentum_cumulative"]
    managed_cumul = backtest_results["managed_cumulative"]
    mom_dd = (mom_cumul - mom_cumul.expanding().max()) / mom_cumul.expanding().max()
    managed_dd = (managed_cumul - managed_cumul.expanding().max()) / managed_cumul.expanding().max()
    axes[1].fill_between(dates, 0, mom_dd * 100, alpha=0.3, color="blue", label="Standard Momentum")
    axes[1].fill_between(
        dates, 0, managed_dd * 100, alpha=0.3, color="green", label="Crash-Managed"
    )
    axes[1].set_ylabel("Drawdown (%)", fontsize=11)
    axes[1].set_title("Strategy Drawdowns", fontsize=12, fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    if "loser_beta" in crash_risk_data.columns:
        axes[2].plot(
            crash_risk_data.index,
            crash_risk_data["loser_beta"],
            color="red",
            linewidth=1.5,
            alpha=0.7,
        )
        axes[2].axhline(
            y=1.5, color="red", linestyle="--", linewidth=2, label="High Beta Threshold (1.5)"
        )
        axes[2].axhline(
            y=1.0, color="orange", linestyle="--", linewidth=1, label="Neutral Beta (1.0)"
        )
        axes[2].set_ylabel("Loser Portfolio Beta", fontsize=11)
        axes[2].set_title(
            "Loser Portfolio Market Beta (Crash Risk Indicator)", fontsize=12, fontweight="bold"
        )
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    if "crash_risk_score" in crash_risk_data.columns:
        ax4a = axes[3]
        ax4b = ax4a.twinx()
        ax4a.fill_between(
            crash_risk_data.index,
            0,
            crash_risk_data["crash_risk_score"],
            alpha=0.3,
            color="red",
            label="Crash Risk Score",
        )
        ax4a.plot(
            crash_risk_data.index, crash_risk_data["crash_risk_score"], color="red", linewidth=1.5
        )
        ax4b.plot(
            crash_risk_data.index,
            crash_risk_data["exposure"],
            color="green",
            linewidth=2,
            label="Momentum Exposure",
        )
        ax4a.set_ylabel("Crash Risk Score (0-3)", fontsize=11, color="red")
        ax4b.set_ylabel("Exposure Level", fontsize=11, color="green")
        ax4a.set_title("Crash Risk Score and Dynamic Exposure", fontsize=12, fontweight="bold")
        ax4a.legend(loc="upper left")
        ax4b.legend(loc="upper right")
        ax4a.grid(True, alpha=0.3)
    rolling_window = 252
    mom_rolling_sharpe = (
        backtest_results["momentum_return"].rolling(rolling_window).mean()
        / backtest_results["momentum_return"].rolling(rolling_window).std()
        * np.sqrt(252)
    )
    managed_rolling_sharpe = (
        backtest_results["managed_return"].rolling(rolling_window).mean()
        / backtest_results["managed_return"].rolling(rolling_window).std()
        * np.sqrt(252)
    )
    axes[4].plot(
        dates, mom_rolling_sharpe, color="blue", linewidth=1.5, alpha=0.7, label="Standard Momentum"
    )
    axes[4].plot(
        dates,
        managed_rolling_sharpe,
        color="green",
        linewidth=1.5,
        alpha=0.7,
        label="Crash-Managed",
    )
    axes[4].axhline(y=0, color="black", linestyle="-", linewidth=1)
    axes[4].set_ylabel("Sharpe Ratio", fontsize=11)
    axes[4].set_title("Rolling 1-Year Sharpe Ratio", fontsize=12, fontweight="bold")
    axes[4].legend()
    axes[4].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    print("=" * 80)
    print("DANIEL-MOSKOWITZ MOMENTUM CRASH RISK MODEL")
    print("=" * 80)
    print("\nObjective: Dynamically manage momentum strategy exposure")
    print("Methodology: Predict crash risk using loser beta and market conditions")
    print("Application: Momentum strategies with crash risk management\n")
    try:
        tickers = fetch_stock_list(limit=UNIVERSE_SIZE)
        prices = fetch_prices_for_universe(tickers, START_DATE, END_DATE)
        market_prices = fetch_market_data(BENCHMARK_TICKER, START_DATE, END_DATE)
        common_dates = prices.index.intersection(market_prices.index)
        prices = prices.loc[common_dates]
        market_prices = market_prices.loc[common_dates]
        market_returns = market_prices.pct_change()
        momentum_signals = calculate_momentum_signals(prices, FORMATION_PERIOD, SKIP_MONTH)
        portfolio_assignments = construct_momentum_portfolios(momentum_signals, NUM_PORTFOLIOS)
        print("\nCalculating loser portfolio characteristics...")
        loser_returns = calculate_portfolio_returns(prices, portfolio_assignments, portfolio_id=1)
        loser_beta = calculate_rolling_beta(loser_returns, market_returns, BETA_WINDOW)
        print("\nCalculating crash risk indicators...")
        market_return_rolling = market_returns.rolling(MARKET_RETURN_WINDOW).sum()
        market_vol_rolling = market_returns.rolling(VOLATILITY_WINDOW).std() * np.sqrt(252)
        crash_risk_data = calculate_crash_risk_indicator(
            loser_beta,
            market_return_rolling,
            market_vol_rolling,
            BETA_THRESHOLD_HIGH,
            MARKET_RETURN_THRESHOLD,
            VOLATILITY_THRESHOLD,
        )
        crash_risk_data["exposure"] = determine_exposure(
            crash_risk_data["crash_risk_score"], low_threshold=1, high_threshold=2
        )
        print("\n" + "=" * 80)
        print("BACKTESTING")
        print("=" * 80)
        backtest_standard = backtest_momentum_strategy(
            prices,
            portfolio_assignments,
            market_returns,
            crash_risk_data,
            use_crash_management=False,
            holding_period=HOLDING_PERIOD,
            commission_rate=COMMISSION_RATE,
        )
        backtest_managed = backtest_momentum_strategy(
            prices,
            portfolio_assignments,
            market_returns,
            crash_risk_data,
            use_crash_management=True,
            holding_period=HOLDING_PERIOD,
            commission_rate=COMMISSION_RATE,
        )
        combined_results = pd.DataFrame(index=backtest_standard.index)
        combined_results["momentum_return"] = backtest_standard["momentum_return"]
        combined_results["managed_return"] = backtest_managed["managed_return"]
        combined_results["momentum_cumulative"] = backtest_standard["momentum_cumulative"]
        combined_results["managed_cumulative"] = backtest_managed["managed_cumulative"]
        combined_results["market_cumulative"] = backtest_standard["market_cumulative"]
        print("\n" + "=" * 80)
        print("PERFORMANCE COMPARISON")
        print("=" * 80)
        mom_metrics = calculate_performance_metrics(backtest_standard["momentum_return"].dropna())
        managed_metrics = calculate_performance_metrics(backtest_managed["managed_return"].dropna())
        market_metrics = calculate_performance_metrics(
            market_returns.reindex(combined_results.index, fill_value=0)
        )
        comparison = pd.DataFrame(
            {
                "Standard Momentum": mom_metrics,
                "Crash-Managed Momentum": managed_metrics,
                "Market (SPY)": market_metrics,
            }
        )
        print("\n" + comparison.to_string())
        print("\n" + "=" * 80)
        print("CRASH RISK MANAGEMENT IMPACT")
        print("=" * 80)
        sharpe_improvement = (
            (managed_metrics["Sharpe Ratio"] - mom_metrics["Sharpe Ratio"])
            / abs(mom_metrics["Sharpe Ratio"])
            * 100
        )
        dd_improvement = (
            (managed_metrics["Max Drawdown"] - mom_metrics["Max Drawdown"])
            / abs(mom_metrics["Max Drawdown"])
            * 100
        )
        print(f"\nSharpe Ratio Improvement: {sharpe_improvement:+.1f}%")
        print(f"Max Drawdown Improvement: {dd_improvement:+.1f}%")
        print("\n" + "=" * 80)
        print("CRASH RISK PERIODS")
        print("=" * 80)
        high_risk_periods = crash_risk_data[crash_risk_data["crash_risk_score"] >= 2]
        print(f"\nHigh crash risk periods detected: {len(high_risk_periods)}")
        if len(high_risk_periods) > 0:
            print(f"\nSample high-risk dates:")
            print(
                high_risk_periods.head(10)[
                    ["loser_beta", "market_return", "crash_risk_score", "exposure"]
                ].to_string()
            )
        plot_backtest_results(combined_results, crash_risk_data, BENCHMARK_TICKER)
        print("\n" + "=" * 80)
        print("ANALYSIS COMPLETE")
        print("=" * 80)
    except Exception as e:
        print(f"\nERROR: {str(e)}")
        import traceback

        traceback.print_exc()
